# Superstore Orders - Column-Level Profile

- **File:** `Data /SuperStoreOrders.csv`
- **Shape:** 51,290 rows x 21 original columns
- **Grain:** one product line per row, not one order per row
- **Order dates:** 2011-01-01 to 2014-12-31
- **Ship dates:** 2011-01-03 to 2015-01-07
- **Coverage:** 147 countries, 7 markets, 13 regions, 3 categories, 17 sub-categories
- **Purpose:** establish the schema, numeric and categorical profiles, data-quality constraints, and business signals that the agentic AI analytics workflow must respect.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 55)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("../Data /SuperStoreOrders.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("Data /SuperStoreOrders.csv")

df = pd.read_csv(DATA_PATH, thousands=",", dtype={"year": "Int64"})
df["order_date"] = pd.to_datetime(df["order_date"], dayfirst=True, format="mixed")
df["ship_date"] = pd.to_datetime(df["ship_date"], dayfirst=True, format="mixed")
df["ship_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["year_check"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_quarter"] = df["order_date"].dt.quarter
df["order_weekday"] = df["order_date"].dt.day_name()
df["profit_margin"] = df["profit"] / df["sales"].replace(0, np.nan)
df["freight_ratio"] = df["shipping_cost"] / df["sales"].replace(0, np.nan)
df["discount_band"] = pd.cut(
    df["discount"],
    bins=[-0.001, 0.0001, 0.1, 0.2, 0.3, 0.4, 0.4999, 0.5999, 1],
    labels=["0%", "0-10%", "10-20%", "20-30%", "30-40%", "40-50%", "50-60%", "60%+"],
    include_lowest=True
)
df["order_key"] = (
    df["order_id"] + "|" + df["customer_name"] + "|" + df["order_date"].dt.strftime("%Y-%m-%d")
)

display(df.head(3))

,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year,ship_days,year_check,order_month,order_quarter,order_weekday,profit_margin,freight_ratio,discount_band,order_key
0,AG-2011-2040,2011-01-01,2011-01-06,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,OFF-TEN-10000025,Office Supplies,Storage,"Tenex Lockers, Blue",408,2,0.00,106.14,35.46,Medium,2011,5,2011,1,1,Saturday,0.26,0.09,0%,AG-2011-2040|Toby Braunhardt|2011-01-01
1,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,OFF-SU-10000618,Office Supplies,Supplies,"Acme Trimmer, High Speed",120,3,0.10,36.04,9.72,Medium,2011,7,2011,1,1,Saturday,0.30,0.08,0-10%,IN-2011-47883|Joseph Holt|2011-01-01
2,HU-2011-1220,2011-01-01,2011-01-05,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,OFF-TEN-10001585,Office Supplies,Storage,"Tenex Box, Single Width",66,4,0.00,29.64,8.17,High,2011,4,2011,1,1,Saturday,0.45,0.12,0%,HU-2011-1220|Annie Thurman|2011-01-01


## 1. Structure, data types and missingness

All 21 source columns are populated. The CSV is clean at the row level, but it is **not clean as an order table**: `order_id` is repeated for every product line and also collides across customers in a small number of cases. Two engineered ratio cells are missing because one line has zero sales.

| Column | Type | Role | Notes |
|---|---|---|---|
| `order_id` | identifier | order-line key | 25,035 distinct values; not guaranteed unique |
| `order_date` | date | order timing | mixed `/` and `-` formats; day-first |
| `ship_date` | date | fulfilment timing | mixed `/` and `-` formats; day-first |
| `ship_mode` | categorical | fulfilment | 4 modes |
| `customer_name` | identifier | customer label | only 795 names; reused across countries, not a customer ID |
| `segment` | categorical | customer segment | Consumer, Corporate, Home Office |
| `state` | categorical | geography | 1,094 values; 16 names occur in multiple countries |
| `country` | categorical | geography | 147 countries |
| `market` | categorical | geography | 7 markets |
| `region` | categorical | geography | 13 regions; nested within market |
| `product_id` | identifier | product key | 10,292 values; preferred product key |
| `category` | categorical | product hierarchy | 3 categories |
| `sub_category` | categorical | product hierarchy | 17 sub-categories |
| `product_name` | label | product label | 3,788 labels; 1,944 labels map to multiple IDs |
| `sales` | numeric | line revenue | comma thousands separators in raw CSV |
| `quantity` | numeric | units | 1 to 14 |
| `discount` | numeric | line discount rate | 0 to 0.85 |
| `profit` | numeric | line profit | negative values common |
| `shipping_cost` | numeric | fulfilment cost | 0 to 933.57 |
| `order_priority` | categorical | order handling | Low, Medium, High, Critical |
| `year` | temporal | order year | exactly matches `order_date.year` |

In [2]:
original_columns = [
    "order_id", "order_date", "ship_date", "ship_mode", "customer_name", "segment",
    "state", "country", "market", "region", "product_id", "category", "sub_category",
    "product_name", "sales", "quantity", "discount", "profit", "shipping_cost",
    "order_priority", "year"
]
engineered_columns = [
    "ship_days", "year_check", "order_month", "order_quarter", "order_weekday",
    "profit_margin", "freight_ratio", "discount_band", "order_key"
]

def structure_table(columns):
    return pd.DataFrame({
        "dtype": df[columns].dtypes.astype(str),
        "non_null": df[columns].notna().sum(),
        "missing": df[columns].isna().sum(),
        "missing_pct": (df[columns].isna().mean() * 100).round(2),
        "unique": df[columns].nunique(dropna=True),
    })

display(structure_table(original_columns))
display(structure_table(engineered_columns))

print(f"Rows: {len(df):,}")
print(f"Original columns: {len(original_columns)}")
print(f"Engineered columns: {len(engineered_columns)}")
print(f"Missing cells: {int(df.isna().sum().sum()):,}")
print(f"Exact duplicate rows: {int(df.duplicated().sum()):,}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:,.2f} MB")
print(f"Order date range: {df['order_date'].min():%Y-%m-%d} -> {df['order_date'].max():%Y-%m-%d}")
print(f"Ship date range: {df['ship_date'].min():%Y-%m-%d} -> {df['ship_date'].max():%Y-%m-%d}")

,dtype,non_null,missing,missing_pct,unique
order_id,str,51290,0,0.00,25035
order_date,datetime64[us],51290,0,0.00,1430
ship_date,datetime64[us],51290,0,0.00,1464
ship_mode,str,51290,0,0.00,4
customer_name,str,51290,0,0.00,795
segment,str,51290,0,0.00,3
state,str,51290,0,0.00,1094
country,str,51290,0,0.00,147
market,str,51290,0,0.00,7
region,str,51290,0,0.00,13


,dtype,non_null,missing,missing_pct,unique
ship_days,int64,51290,0,0.00,8
year_check,int32,51290,0,0.00,4
order_month,int32,51290,0,0.00,12
order_quarter,int32,51290,0,0.00,4
order_weekday,str,51290,0,0.00,7
profit_margin,float64,51289,1,0.00,29000
freight_ratio,float64,51289,1,0.00,35696
discount_band,category,51290,0,0.00,8
order_key,str,51290,0,0.00,25754


Rows: 51,290
Original columns: 21
Engineered columns: 9
Missing cells: 2
Exact duplicate rows: 0
Memory usage: 50.62 MB
Order date range: 2011-01-01 -> 2014-12-31
Ship date range: 2011-01-03 -> 2015-01-07


## 2. Numeric columns - mean, median, mode and spread

`sales` is strongly right-skewed: the median line is only $85, while the mean is $246.50 and the maximum is $22,638. `profit` is even more dispersed, with a positive mean but a long negative tail. `discount` is zero for more than half of all lines.

In [3]:
numeric_columns = ["sales", "quantity", "discount", "profit", "shipping_cost", "ship_days", "year"]

def numeric_profile(columns):
    records = []
    for col in columns:
        s = df[col].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        mode = s.mode()
        records.append({
            "column": col,
            "count": len(s),
            "mean": s.mean(),
            "median": s.median(),
            "mode": mode.iloc[0] if not mode.empty else np.nan,
            "std": s.std(),
            "min": s.min(),
            "q1": q1,
            "q3": q3,
            "max": s.max(),
            "iqr": q3 - q1,
            "skew": s.skew(),
            "kurtosis": s.kurt(),
            "zeros": int((s == 0).sum()),
            "negatives": int((s < 0).sum())
        })
    return pd.DataFrame(records).set_index("column")

display(numeric_profile(numeric_columns).round(2))

def outlier_summary(columns):
    records = []
    for col in columns:
        s = df[col].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        records.append({
            "column": col,
            "lower_fence": lower,
            "upper_fence": upper,
            "low_outliers": int((s < lower).sum()),
            "high_outliers": int((s > upper).sum()),
            "outlier_share_pct": (s.lt(lower) | s.gt(upper)).mean() * 100
        })
    return pd.DataFrame(records).set_index("column")

display(outlier_summary(["sales", "profit", "shipping_cost", "quantity", "discount", "ship_days"]).round(2))

,count,mean,median,mode,std,min,q1,q3,max,iqr,skew,kurtosis,zeros,negatives
column,,,,,,,,,,,,,,
sales,51290,246.50,85.00,13.00,487.57,0.00,31.00,251.00,"22,638.00",220.00,8.14,176.72,1,0
quantity,51290,3.48,3.00,2.00,2.28,1.00,2.00,5.00,14.00,3.00,1.36,2.28,0,0
discount,51290,0.14,0.00,0.00,0.21,0.00,0.00,0.20,0.85,0.20,1.39,0.72,29009,0
profit,51290,28.64,9.24,0.00,174.42,"-6,599.98",0.00,36.81,"8,399.98",36.81,4.16,290.90,668,12543
shipping_cost,51290,26.38,7.79,0.86,57.30,0.00,2.61,24.45,933.57,21.84,5.86,50.02,2,0
ship_days,51290,3.97,4.00,4.00,1.73,0.00,3.00,5.00,7.00,2.00,-0.43,-0.25,2600,0
year,51290,"2,012.78","2,013.00","2,014.00",1.10,"2,011.00","2,012.00","2,014.00","2,014.00",2.00,-0.34,-1.22,0,0


,lower_fence,upper_fence,low_outliers,high_outliers,outlier_share_pct
column,,,,,
sales,-299.00,581.00,0,5655,11.03
profit,-55.22,92.02,3525,6230,19.02
shipping_cost,-30.15,57.21,0,5909,11.52
quantity,-2.50,9.50,0,877,1.71
discount,-0.30,0.50,0,4172,8.13
ship_days,0.00,8.00,0,0,0.00


## 3. Date columns - parse quality, range and fulfilment gaps

Date strings are a real ingestion hazard. The raw file mixes `D/M/YYYY` and `DD-MM-YYYY`. Parsing day-first produces zero negative shipping gaps and a maximum gap of 7 days. Parsing month-first produces 9,687 negative gaps and 14,324 gaps longer than 30 days, so **day-first is the only defensible parser** for this file.

In [4]:
date_rows = []
for col in ["order_date", "ship_date"]:
    s = df[col]
    date_rows.append({
        "column": col,
        "non_null": s.notna().sum(),
        "min": s.min().strftime("%Y-%m-%d"),
        "max": s.max().strftime("%Y-%m-%d"),
        "span_days": (s.max() - s.min()).days,
        "unique_dates": s.nunique(),
        "median_date": s.median().strftime("%Y-%m-%d"),
        "most_common_date": s.mode().iloc[0].strftime("%Y-%m-%d"),
    })
date_profile = pd.DataFrame(date_rows).set_index("column")
display(date_profile)

raw_dates = pd.read_csv(DATA_PATH, usecols=["order_date", "ship_date"], dtype=str)
raw_order_dates = raw_dates["order_date"]
raw_ship_dates = raw_dates["ship_date"]
naive_order = pd.to_datetime(raw_order_dates, dayfirst=False, format="mixed")
naive_ship = pd.to_datetime(raw_ship_dates, dayfirst=False, format="mixed")
naive_ship_days = (naive_ship - naive_order).dt.days

parse_audit = pd.Series({
    "order_date slash values": int(raw_order_dates.str.contains("/", regex=False).sum()),
    "order_date dash values": int(raw_order_dates.str.contains("-", regex=False).sum()),
    "ship_date slash values": int(raw_ship_dates.str.contains("/", regex=False).sum()),
    "ship_date dash values": int(raw_ship_dates.str.contains("-", regex=False).sum()),
    "parse failures day-first": int(df["order_date"].isna().sum() + df["ship_date"].isna().sum()),
    "negative ship gaps day-first": int((df["ship_days"] < 0).sum()),
    "gaps over 30 days day-first": int((df["ship_days"] > 30).sum()),
    "negative ship gaps month-first": int((naive_ship_days < 0).sum()),
    "gaps over 30 days month-first": int((naive_ship_days > 30).sum()),
    "year column mismatches": int((df["year"] != df["year_check"]).sum()),
})
display(parse_audit.to_frame("value"))

ship_day_view = df["ship_days"].value_counts().sort_index().to_frame("line_items")
ship_day_view["share_pct"] = ship_day_view["line_items"] / len(df) * 100
display(ship_day_view)

,non_null,min,max,span_days,unique_dates,median_date,most_common_date
column,,,,,,,
order_date,51290,2011-01-01,2014-12-31,1460,1430,2013-07-08,2014-06-18
ship_date,51290,2011-01-03,2015-01-07,1465,1464,2013-07-12,2014-11-22


,value
order_date slash values,20067
order_date dash values,31223
ship_date slash values,19834
ship_date dash values,31456
parse failures day-first,0
negative ship gaps day-first,0
gaps over 30 days day-first,0
negative ship gaps month-first,9687
gaps over 30 days month-first,14324
year column mismatches,0


,line_items,share_pct
ship_days,,
0,2600,5.07
1,1662,3.24
2,7026,13.70
3,5035,9.82
4,14434,28.14
5,11221,21.88
6,6255,12.20
7,3057,5.96


## 4. Categorical and identifier columns

Low-cardinality fields are clean and compact. High-cardinality fields carry important caveats: `customer_name` is not a customer entity, `product_name` is not a product key, and `state` is not globally unique without `country`.

In [5]:
low_cardinality = ["ship_mode", "segment", "market", "region", "category", "sub_category", "order_priority", "year"]

def categorical_profile(columns):
    records = []
    for col in columns:
        vc = df[col].value_counts(dropna=False)
        records.append({
            "column": col,
            "non_null": df[col].notna().sum(),
            "unique": df[col].nunique(dropna=True),
            "top_value": vc.index[0],
            "top_count": int(vc.iloc[0]),
            "top_share_pct": vc.iloc[0] / len(df) * 100
        })
    return pd.DataFrame(records).set_index("column")

display(categorical_profile(low_cardinality).round(2))

identifier_columns = ["order_id", "customer_name", "state", "country", "product_id", "product_name"]

def identifier_profile(columns):
    records = []
    for col in columns:
        vc = df[col].value_counts(dropna=False)
        records.append({
            "column": col,
            "rows": len(df),
            "unique": df[col].nunique(dropna=True),
            "rows_per_unique": len(df) / df[col].nunique(dropna=True),
            "max_repeat": int(vc.iloc[0]),
            "top_value": vc.index[0]
        })
    return pd.DataFrame(records).set_index("column")

display(identifier_profile(identifier_columns).round(2))

order_rollup = df.groupby("order_id").agg(
    lines=("sales", "size"),
    customers=("customer_name", "nunique"),
    countries=("country", "nunique"),
    states=("state", "nunique"),
    ship_modes=("ship_mode", "nunique"),
    categories=("category", "nunique")
)
composite_rollup = df.groupby("order_key").agg(
    countries=("country", "nunique"),
    states=("state", "nunique"),
    ship_modes=("ship_mode", "nunique")
)
customer_country_count = df.groupby("customer_name")["country"].nunique()
product_name_id_count = df.groupby("product_name")["product_id"].nunique()
state_country_count = df.groupby("state")["country"].nunique()

grain_checks = pd.Series({
    "line items": len(df),
    "distinct order_id values": df["order_id"].nunique(),
    "distinct composite order keys": df["order_key"].nunique(),
    "order IDs spanning multiple customers": int((order_rollup["customers"] > 1).sum()),
    "order IDs spanning multiple countries": int((order_rollup["countries"] > 1).sum()),
    "order IDs spanning multiple ship modes": int((order_rollup["ship_modes"] > 1).sum()),
    "composite keys spanning multiple countries": int((composite_rollup["countries"] > 1).sum()),
    "composite keys spanning multiple ship modes": int((composite_rollup["ship_modes"] > 1).sum()),
    "duplicate order_id + product_id rows": int(df.duplicated(["order_id", "product_id"]).sum()),
    "distinct customer_name values": df["customer_name"].nunique(),
    "customer names appearing in multiple countries": int((customer_country_count > 1).sum()),
    "maximum countries for one customer name": int(customer_country_count.max()),
    "distinct product_id values": df["product_id"].nunique(),
    "product names mapping to multiple product IDs": int((product_name_id_count > 1).sum()),
    "distinct country + state pairs": df[["country", "state"]].drop_duplicates().shape[0],
    "state names appearing in multiple countries": int((state_country_count > 1).sum()),
})
display(grain_checks.to_frame("value"))

,non_null,unique,top_value,top_count,top_share_pct
column,,,,,
ship_mode,51290,4,Standard Class,30775,60.00
segment,51290,3,Consumer,26518,51.70
market,51290,7,APAC,11002,21.45
region,51290,13,Central,11117,21.67
category,51290,3,Office Supplies,31273,60.97
sub_category,51290,17,Binders,6152,11.99
order_priority,51290,4,Medium,29433,57.39
year,51290,4,2014,17531,34.18


,rows,unique,rows_per_unique,max_repeat,top_value
column,,,,,
order_id,51290,25035,2.05,14,CA-2014-100111
customer_name,51290,795,64.52,108,Muhammed Yedwab
state,51290,1094,46.88,2001,California
country,51290,147,348.91,9994,United States
product_id,51290,10292,4.98,35,OFF-AR-10003651
product_name,51290,3788,13.54,227,Staples


,value
line items,51290
distinct order_id values,25035
distinct composite order keys,25754
order IDs spanning multiple customers,659
order IDs spanning multiple countries,58
order IDs spanning multiple ship modes,398
composite keys spanning multiple countries,0
composite keys spanning multiple ship modes,0
duplicate order_id + product_id rows,38
distinct customer_name values,795


## 5. Sales, profit, discount and shipping distributions

The headline economics are healthy overall but fragile at the line level: total sales are $12.64M, total profit is $1.47M, and the aggregate margin is 11.62%. However, 24.46% of lines lose money, and high discounts are almost perfectly associated with losses.

In [6]:
distribution_metrics = ["sales", "quantity", "discount", "profit", "profit_margin", "shipping_cost", "freight_ratio", "ship_days"]
quantiles = [0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1]
display(df[distribution_metrics].quantile(quantiles).T.round(4))

profit_bridge = pd.Series({
    "line items": len(df),
    "sales": df["sales"].sum(),
    "profit": df["profit"].sum(),
    "profit_margin_pct": df["profit"].sum() / df["sales"].sum() * 100,
    "loss-making lines": int((df["profit"] < 0).sum()),
    "loss-making share_pct": (df["profit"] < 0).mean() * 100,
    "loss amount": df.loc[df["profit"] < 0, "profit"].sum(),
    "profitable lines": int((df["profit"] > 0).sum()),
    "profit amount": df.loc[df["profit"] > 0, "profit"].sum(),
    "zero-profit lines": int((df["profit"] == 0).sum()),
})
display(profit_bridge.to_frame("value").round(2))

discount_view = df.groupby("discount_band", observed=True).agg(
    line_items=("sales", "size"),
    sales=("sales", "sum"),
    profit=("profit", "sum"),
    avg_discount_pct=("discount", lambda s: s.mean() * 100),
    loss_share_pct=("profit", lambda s: (s < 0).mean() * 100)
)
discount_view["sales_share_pct"] = discount_view["sales"] / df["sales"].sum() * 100
discount_view["margin_pct"] = discount_view["profit"] / discount_view["sales"] * 100
discount_view = discount_view[["line_items", "sales", "sales_share_pct", "profit", "margin_pct", "loss_share_pct", "avg_discount_pct"]]
display(discount_view.round(2))

corr_cols = ["sales", "quantity", "discount", "profit", "profit_margin", "shipping_cost", "freight_ratio", "ship_days"]
print(df[corr_cols].corr().round(3).to_string(float_format=lambda x: f"{x:.3f}"))

scenarios = pd.DataFrame({
    "rows": [
        len(df),
        int((df["profit"] >= -df["sales"]).sum()),
        int((df["discount"] < 0.5).sum()),
        int(((df["profit"] >= -df["sales"]) & (df["discount"] < 0.5)).sum())
    ],
    "sales": [
        df["sales"].sum(),
        df.loc[df["profit"] >= -df["sales"], "sales"].sum(),
        df.loc[df["discount"] < 0.5, "sales"].sum(),
        df.loc[(df["profit"] >= -df["sales"]) & (df["discount"] < 0.5), "sales"].sum()
    ],
    "profit": [
        df["profit"].sum(),
        df.loc[df["profit"] >= -df["sales"], "profit"].sum(),
        df.loc[df["discount"] < 0.5, "profit"].sum(),
        df.loc[(df["profit"] >= -df["sales"]) & (df["discount"] < 0.5), "profit"].sum()
    ],
    "loss_share_pct": [
        (df["profit"] < 0).mean() * 100,
        (df.loc[df["profit"] >= -df["sales"], "profit"] < 0).mean() * 100,
        (df.loc[df["discount"] < 0.5, "profit"] < 0).mean() * 100,
        (df.loc[(df["profit"] >= -df["sales"]) & (df["discount"] < 0.5), "profit"] < 0).mean() * 100
    ]
}, index=["All lines", "Exclude profit below -sales", "Exclude discount >= 50%", "Core: both filters"])
scenarios["margin_pct"] = scenarios["profit"] / scenarios["sales"] * 100
display(scenarios.round(2))

anomaly_checks = pd.Series({
    "profit < 0": int((df["profit"] < 0).sum()),
    "profit < -sales": int((df["profit"] < -df["sales"]).sum()),
    "profit < -2 * sales": int((df["profit"] < -2 * df["sales"]).sum()),
    "profit margin < -100%": int((df["profit_margin"] < -1).sum()),
    "shipping_cost > sales": int((df["shipping_cost"] > df["sales"]).sum()),
    "sales == 0": int((df["sales"] == 0).sum()),
    "discount >= 50%": int((df["discount"] >= 0.5).sum()),
})
display(anomaly_checks.to_frame("line_items"))

worst_margin = df.nsmallest(8, "profit_margin")[
    ["order_id", "sales", "discount", "profit", "profit_margin", "shipping_cost", "category", "sub_category", "country"]
].copy()
worst_margin["profit_margin_pct"] = worst_margin.pop("profit_margin") * 100
display(worst_margin.round(2))

,0.00,0.01,0.05,0.25,0.50,0.75,0.90,0.95,0.99,1.00
sales,0.00,4.00,9.00,31.00,85.00,251.00,632.00,"1,016.00","2,301.00","22,638.00"
quantity,1.00,1.00,1.00,2.00,3.00,5.00,7.00,8.00,11.00,14.00
discount,0.00,0.00,0.00,0.00,0.00,0.20,0.50,0.60,0.70,0.85
profit,"-6,599.98",-351.51,-83.88,0.00,9.24,36.81,112.68,211.50,588.79,"8,399.98"
profit_margin,-4.73,-1.83,-0.92,0.00,0.17,0.33,0.44,0.47,0.50,0.58
shipping_cost,0.00,0.20,0.61,2.61,7.79,24.45,65.13,111.41,286.75,933.57
freight_ratio,0.00,0.01,0.03,0.06,0.09,0.14,0.20,0.25,0.36,0.59
ship_days,0.00,0.00,0.00,3.00,4.00,5.00,6.00,7.00,7.00,7.00


,value
line items,"51,290.00"
sales,"12,642,905.00"
profit,"1,469,034.82"
profit_margin_pct,11.62
loss-making lines,"12,543.00"
loss-making share_pct,24.46
loss amount,"-920,357.39"
profitable lines,"38,079.00"
profit amount,"2,389,392.21"
zero-profit lines,668.00


,line_items,sales,sales_share_pct,profit,margin_pct,loss_share_pct,avg_discount_pct
discount_band,,,,,,,
0%,29009,6992734,55.31,"1,770,695.27",25.32,0.00,0.00
0-10%,4679,1962633,15.52,"339,766.79",17.31,19.23,8.94
10-20%,6274,1757296,13.90,"173,254.84",9.86,23.32,19.22
20-30%,967,382540,3.03,"-21,155.61",-5.53,62.15,27.36
30-40%,3400,701367,5.55,"-166,115.45",-23.68,80.21,39.69
40-50%,1156,150566,1.19,"-56,199.49",-37.33,92.04,45.82
50-60%,1655,338597,2.68,"-168,094.10",-49.64,98.97,50.08
60%+,4150,357172,2.83,"-403,117.44",-112.86,100.00,65.86


               sales  quantity  discount  profit  profit_margin  shipping_cost  freight_ratio  ship_days
sales          1.000     0.314    -0.087   0.486          0.075          0.768         -0.016     -0.005
quantity       0.314     1.000    -0.020   0.105          0.050          0.273         -0.011      0.004
discount      -0.087    -0.020     1.000  -0.316         -0.846         -0.079         -0.001     -0.002
profit         0.486     0.105    -0.316   1.000          0.357          0.357         -0.005      0.002
profit_margin  0.075     0.050    -0.846   0.357          1.000          0.068          0.003      0.000
shipping_cost  0.768     0.273    -0.079   0.357          0.068          1.000          0.278     -0.143
freight_ratio -0.016    -0.011    -0.001  -0.005          0.003          0.278          1.000     -0.481
ship_days     -0.005     0.004    -0.002   0.002          0.000         -0.143         -0.481      1.000


,rows,sales,profit,loss_share_pct,margin_pct
All lines,51290,12642905,"1,469,034.82",24.46,11.62
Exclude profit below -sales,49030,12443871,"1,765,882.33",20.97,14.19
Exclude discount >= 50%,45485,11947136,"2,040,246.36",14.85,17.08
Core: both filters,45485,11947136,"2,040,246.36",14.85,17.08


,line_items
profit < 0,12543
profit < -sales,2260
profit < -2 * sales,378
profit margin < -100%,2259
shipping_cost > sales,1
sales == 0,1
discount >= 50%,5805


,order_id,sales,discount,profit,shipping_cost,category,sub_category,country,profit_margin_pct
18023,IT-2012-5588535,242,0.85,"-1,144.12",23.38,Furniture,Tables,Germany,-472.78
11874,ID-2012-56913,63,0.80,-244.39,6.07,Furniture,Tables,Pakistan,-387.91
9369,IT-2012-2732419,70,0.80,-271.24,10.74,Furniture,Furnishings,United Kingdom,-387.48
2235,ID-2011-13436,127,0.80,-488.77,14.94,Furniture,Tables,Pakistan,-384.86
50062,CA-2014-165099,1,0.80,-3.76,0.14,Office Supplies,Appliances,United States,-375.84
22273,ID-2013-10713,108,0.80,-390.26,9.62,Furniture,Tables,Pakistan,-361.36
12082,IT-2012-2675422,555,0.85,"-1,924.54",62.75,Furniture,Tables,Germany,-346.76
23362,ID-2013-34177,100,0.80,-344.41,7.72,Furniture,Tables,South Korea,-344.41


## 6. Contribution and profitability by business dimension

Technology is the largest sales category (37.53%) and carries the strongest margin among the three categories (13.99%). Furniture has the weakest margin (6.98%) because Tables loses money. At market level, APAC is the largest region, while EMEA is the weakest large market.

In [7]:
def performance(columns):
    out = df.groupby(columns, dropna=False).agg(
        line_items=("sales", "size"),
        sales=("sales", "sum"),
        profit=("profit", "sum"),
        avg_discount=("discount", "mean"),
        loss_share=("profit", lambda s: (s < 0).mean()),
        shipping_cost=("shipping_cost", "sum")
    )
    out["sales_share_pct"] = out["sales"] / df["sales"].sum() * 100
    out["margin_pct"] = out["profit"] / out["sales"] * 100
    out["loss_share_pct"] = out["loss_share"] * 100
    out["avg_discount_pct"] = out["avg_discount"] * 100
    out["freight_ratio_pct"] = out["shipping_cost"] / out["sales"] * 100
    return out.drop(columns=["avg_discount", "loss_share"])

category_view = performance(["category"]).sort_values("sales", ascending=False)
subcategory_view = performance(["sub_category"]).sort_values("sales", ascending=False)
segment_view = performance(["segment"]).sort_values("sales", ascending=False)

display(category_view.round(2))
display(subcategory_view.round(2))
display(segment_view.round(2))

,line_items,sales,profit,shipping_cost,sales_share_pct,margin_pct,loss_share_pct,avg_discount_pct,freight_ratio_pct
category,,,,,,,,,
Technology,10141,4744691,"663,778.73","507,048.74",37.53,13.99,23.90,13.53,10.69
Furniture,9876,4110884,"286,782.25","440,320.66",32.52,6.98,31.55,16.81,10.71
Office Supplies,31273,3787330,"518,473.83","405,451.29",29.96,13.69,22.39,13.74,10.71


,line_items,sales,profit,shipping_cost,sales_share_pct,margin_pct,loss_share_pct,avg_discount_pct,freight_ratio_pct
sub_category,,,,,,,,,
Phones,3357,1706874,"216,717.01","184,902.77",13.50,12.70,23.50,14.58,10.83
Copiers,2223,1509439,"258,567.55","159,496.49",11.94,17.13,23.62,11.71,10.57
Chairs,3434,1501682,"141,973.80","164,229.85",11.88,9.45,32.00,16.31,10.94
Bookcases,2411,1466559,"161,924.42","155,482.23",11.60,11.04,29.74,15.38,10.60
Storage,5059,1127124,"108,461.49","120,546.46",8.92,9.62,27.28,13.85,10.70
Appliances,1755,1011081,"141,680.59","108,300.78",8.00,14.01,21.77,14.17,10.71
Machines,1486,779071,"58,867.87","79,135.90",6.16,7.56,30.55,16.96,10.16
Tables,861,757034,"-64,083.39","79,861.46",5.99,-8.47,57.61,29.07,10.55
Accessories,3075,749307,"129,626.31","83,513.58",5.93,17.30,21.33,12.05,11.15


,line_items,sales,profit,shipping_cost,sales_share_pct,margin_pct,loss_share_pct,avg_discount_pct,freight_ratio_pct
segment,,,,,,,,,
Consumer,26518,6508141,"749,239.78","697,300.64",51.48,11.51,24.59,14.36,10.71
Corporate,15429,3824808,"442,785.86","410,474.46",30.25,11.58,24.43,14.29,10.73
Home Office,9343,2309956,"277,009.18","245,045.59",18.27,11.99,24.10,14.09,10.61


In [8]:
market_view = performance(["market"]).sort_values("sales", ascending=False)
region_view = performance(["region"]).sort_values("sales", ascending=False)
ship_view = performance(["ship_mode"]).sort_values("sales", ascending=False)
priority_view = performance(["order_priority"]).sort_values("sales", ascending=False)

ship_view["mean_ship_days"] = df.groupby("ship_mode")["ship_days"].mean()
priority_view["mean_ship_days"] = df.groupby("order_priority")["ship_days"].mean()

display(market_view.round(2))
display(region_view.round(2))
display(ship_view.round(2))
display(priority_view.round(2))

shipping_definition_check = pd.DataFrame({
    "profit_margin_pct": ship_view["margin_pct"],
    "shipping_cost_ratio_pct": ship_view["freight_ratio_pct"],
    "profit_minus_shipping_margin_pct": (ship_view["profit"] - ship_view["shipping_cost"]) / ship_view["sales"] * 100
})
display(shipping_definition_check.round(2))

,line_items,sales,profit,shipping_cost,sales_share_pct,margin_pct,loss_share_pct,avg_discount_pct,freight_ratio_pct
market,,,,,,,,,
APAC,11002,3585833,"437,577.58","387,165.11",28.36,12.20,29.27,14.88,10.80
EU,10000,2938139,"372,829.74","309,422.35",23.24,12.69,21.18,10.31,10.53
US,9994,2297354,"286,397.02","238,173.79",18.17,12.47,18.72,15.62,10.37
LATAM,10294,2164687,"221,643.49","234,138.61",17.12,10.24,26.21,13.55,10.82
EMEA,5029,806184,"43,897.97","88,375.73",6.38,5.45,31.93,19.61,10.96
Africa,4587,783776,"88,871.63","88,139.47",6.20,11.34,22.45,15.67,11.25
Canada,384,66932,"17,817.39","7,405.63",0.53,26.62,0.00,0.00,11.06


,line_items,sales,profit,shipping_cost,sales_share_pct,margin_pct,loss_share_pct,avg_discount_pct,freight_ratio_pct
region,,,,,,,,,
Central,11117,2822399,"311,403.98","296,940.56",22.32,11.03,25.02,13.89,10.52
South,6645,1600960,"140,355.77","168,474.07",12.66,8.77,25.40,14.91,10.52
North,4785,1248192,"194,597.95","131,474.09",9.87,15.59,19.04,9.61,10.53
Oceania,3487,1100207,"121,666.64","120,628.51",8.70,11.06,30.34,15.32,10.96
Southeast Asia,3129,884438,"17,852.33","93,675.84",7.00,2.02,53.31,27.21,10.59
North Asia,2338,848349,"165,578.42","95,030.61",6.71,19.52,9.45,4.87,11.20
EMEA,5029,806184,"43,897.97","88,375.73",6.38,5.45,31.93,19.61,10.96
Africa,4587,783776,"88,871.63","88,139.47",6.20,11.34,22.45,15.67,11.25
Central Asia,2048,752839,"132,480.19","77,830.15",5.95,17.60,13.33,6.75,10.34


,line_items,sales,profit,shipping_cost,sales_share_pct,margin_pct,loss_share_pct,avg_discount_pct,freight_ratio_pct,mean_ship_days
ship_mode,,,,,,,,,,
Standard Class,30775,7578889,"890,596.02","614,630.76",59.95,11.75,24.48,14.22,8.11,5.00
Second Class,10309,2565747,"294,161.06","314,112.62",20.29,11.46,24.07,14.06,12.24,3.23
First Class,7505,1831067,"208,104.68","308,103.25",14.48,11.37,25.08,14.89,16.83,2.18
Same Day,2701,667202,"76,173.07","115,974.06",5.28,11.42,23.92,14.35,17.38,0.04


,line_items,sales,profit,shipping_cost,sales_share_pct,margin_pct,loss_share_pct,avg_discount_pct,freight_ratio_pct,mean_ship_days
order_priority,,,,,,,,,,
Medium,29433,7281105,"864,203.76","542,815.98",57.59,11.87,24.59,14.36,7.46,4.52
High,15501,3807699,"420,373.51","509,547.12",30.12,11.04,24.50,14.33,13.38,3.09
Critical,3932,986258,"125,801.69","234,824.28",7.80,12.76,23.25,13.65,23.81,1.81
Low,2424,567843,"58,655.85","65,633.31",4.49,10.33,24.46,14.22,11.56,6.48


,profit_margin_pct,shipping_cost_ratio_pct,profit_minus_shipping_margin_pct
ship_mode,,,
Standard Class,11.75,8.11,3.64
Second Class,11.46,12.24,-0.78
First Class,11.37,16.83,-5.46
Same Day,11.42,17.38,-5.97


Profit and shipping cost are separate fields. Profit margin is almost flat across ship modes (11.37%-11.75%), while the shipping-cost ratio rises from 8.11% for Standard Class to 17.38% for Same Day. That pattern suggests profit is **not** net of shipping_cost. The file does not document the definition, so any net-profit metric must state its assumption; subtracting shipping_cost mechanically would cut the overall margin from 11.62% to 0.92%.

## 7. Geography and country-level performance

Sales are concentrated: the top 10 countries contribute 62.63% of revenue. The US alone contributes 18.17%, followed by Australia, France, China, Germany and Mexico. Several countries with meaningful volume are structurally loss-making in this dataset, especially Nigeria, Turkey and the Netherlands.

In [9]:
country_view = performance(["country"]).sort_values("sales", ascending=False)
state_country_view = performance(["country", "state"]).sort_values("sales", ascending=False)

display(country_view.head(15).round(2))
weak_countries = country_view[country_view["line_items"] >= 100].sort_values("margin_pct").head(10)
display(weak_countries.round(2))
display(state_country_view.head(15).round(2))

country_concentration = pd.Series({
    "countries": df["country"].nunique(),
    "top_10_countries_sales_share_pct": df.groupby("country")["sales"].sum().nlargest(10).sum() / df["sales"].sum() * 100,
    "countries_with_under_100_lines": int((country_view["line_items"] < 100).sum()),
    "markets": df["market"].nunique(),
    "regions": df["region"].nunique(),
})
display(country_concentration.to_frame("value").round(2))

,line_items,sales,profit,shipping_cost,sales_share_pct,margin_pct,loss_share_pct,avg_discount_pct,freight_ratio_pct
country,,,,,,,,,
United States,9994,2297354,"286,397.02","238,173.79",18.17,12.47,18.72,15.62,10.37
Australia,2837,925257,"105,484.96","100,359.02",7.32,11.40,28.62,14.35,10.85
France,2827,858930,"109,029.00","95,387.81",6.79,12.69,15.78,7.23,11.11
China,1880,700591,"150,683.08","78,957.02",5.54,21.51,2.87,1.43,11.27
Germany,2065,628857,"107,322.82","63,965.18",4.97,17.07,12.20,5.70,10.17
Mexico,2644,622620,"102,818.10","67,659.85",4.92,16.51,12.41,6.16,10.87
India,1555,589664,"129,071.84","61,780.72",4.66,21.89,3.54,1.77,10.48
United Kingdom,1633,528570,"111,900.15","53,580.27",4.18,21.17,12.98,6.57,10.14
Indonesia,1390,404887,"15,608.68","43,948.62",3.20,3.86,58.99,29.73,10.85


,line_items,sales,profit,shipping_cost,sales_share_pct,margin_pct,loss_share_pct,avg_discount_pct,freight_ratio_pct
country,,,,,,,,,
Kazakhstan,101,4602,"-7,099.90",502.27,0.04,-154.28,100.00,70.00,10.91
Nigeria,905,54347,"-80,750.72","6,183.11",0.43,-148.58,100.00,70.00,11.38
Turkey,1378,108524,"-98,447.23","11,664.16",0.86,-90.71,100.00,60.00,10.75
Sweden,203,30490,"-17,519.37","3,238.51",0.24,-57.46,99.01,50.79,10.62
Netherlands,435,77520,"-41,070.07","9,005.75",0.61,-52.98,95.40,48.18,11.62
Ireland,106,16638,"-7,392.38","2,017.26",0.13,-44.43,100.00,50.28,12.12
Venezuela,194,26587,"-11,222.71","2,810.06",0.21,-42.21,81.44,40.94,10.57
South Korea,172,33133,"-12,792.83","3,686.31",0.26,-38.61,87.79,45.29,11.13
Pakistan,249,58873,"-22,446.65","5,833.09",0.47,-38.13,87.55,44.46,9.91


line_items   sales     profit  shipping_cost  sales_share_pct  margin_pct  loss_share_pct  avg_discount_pct  freight_ratio_pct
country        state                                                                                                                                                     
United Kingdom England                           1499  485166  99,907.73      49,133.96             3.84       20.59           13.88              7.02              10.13
United States  California                        2001  457731  76,381.39      48,600.58             3.62       16.69            5.25              7.28              10.62
France         Ile-de-France                      981  317818  44,055.92      35,528.04             2.51       13.86           12.03              5.17              11.18
United States  New York                          1128  310914  74,038.55      31,960.94             2.46       23.81            4.17              5.53              10.28
Australia      New South Wales                    781  270509  45,273.51      28,127.14             2.14       16.74           16.52              8.78              10.40
               Queensland                         717  238311  21,608.75      27,990.10             1.88        9.07           35.56             16.86              11.75
Germany        North Rhine-Westphalia             719  216458  42,347.87      21,252.36             1.71       19.56            7.23              2.94               9.82
United States  Texas                              985  170187 -25,729.36      17,346.40             1.35      -15.12           49.34             37.02              10.19
El Salvador    San Salvador                       615  153648  35,883.38      15,915.12             1.22       23.35            0.16              0.40              10.36
Australia      Victoria                           491  151788  14,929.08      15,150.56             1.20        9.84           29.53             14.30               9.98
Philippines    National Capital                   561  149257 -12,647.48      14,679.75             1.18       -8.47           68.09             34.57               9.84
United States  Washington                         506  138656  33,402.65      13,679.51             1.10       24.09            3.36              6.40               9.87
Mexico         Distrito Federal                   469  128428  20,081.63      14,397.74             1.02       15.64           13.22              6.15              11.21
United States  Pennsylvania                       587  116522 -15,559.96      13,046.39             0.92      -13.35           47.36             32.86              11.20
France         Provence-Alpes-Côte d'Azur         390  114368  16,858.05      12,287.56             0.90       14.74           12.56              5.04              10.74

,value
countries,147.00
top_10_countries_sales_share_pct,62.63
countries_with_under_100_lines,82.00
markets,7.00
regions,13.00


## 8. Time patterns and seasonality

The business grows every year in the file: 2014 contributes 34.01% of all sales and 34.32% of profit. Margin stays in a narrow 11.0%-12.0% band, so growth is volume-led rather than margin-led. November and December are the strongest months; Sunday is by far the weakest weekday.

In [10]:
yearly = df.groupby("year").agg(
    line_items=("sales", "size"),
    orders=("order_key", "nunique"),
    sales=("sales", "sum"),
    profit=("profit", "sum"),
    avg_discount_pct=("discount", lambda s: s.mean() * 100)
)
yearly["margin_pct"] = yearly["profit"] / yearly["sales"] * 100
yearly["avg_order_value"] = yearly["sales"] / yearly["orders"]
yearly["sales_yoy_pct"] = yearly["sales"].pct_change() * 100
yearly["profit_yoy_pct"] = yearly["profit"].pct_change() * 100
yearly["sales_share_pct"] = yearly["sales"] / df["sales"].sum() * 100
display(yearly.round(2))

month_map = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun", 7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
monthly = df.groupby("order_month").agg(
    line_items=("sales", "size"),
    sales=("sales", "sum"),
    profit=("profit", "sum")
)
monthly.index = monthly.index.map(month_map)
monthly["sales_share_pct"] = monthly["sales"] / df["sales"].sum() * 100
monthly["margin_pct"] = monthly["profit"] / monthly["sales"] * 100
display(monthly.round(2))

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_view = df.groupby("order_weekday").agg(
    line_items=("sales", "size"),
    sales=("sales", "sum"),
    profit=("profit", "sum")
).reindex(weekday_order)
weekday_view["sales_share_pct"] = weekday_view["sales"] / df["sales"].sum() * 100
weekday_view["margin_pct"] = weekday_view["profit"] / weekday_view["sales"] * 100
display(weekday_view.round(2))

quarterly = df.groupby("order_quarter").agg(
    line_items=("sales", "size"),
    sales=("sales", "sum"),
    profit=("profit", "sum")
)
quarterly["sales_share_pct"] = quarterly["sales"] / df["sales"].sum() * 100
quarterly["margin_pct"] = quarterly["profit"] / quarterly["sales"] * 100
display(quarterly.round(2))

,line_items,orders,sales,profit,avg_discount_pct,margin_pct,avg_order_value,sales_yoy_pct,profit_yoy_pct,sales_share_pct
year,,,,,,,,,,
2011,8998,4516,2259511,"248,940.81",14.82,11.02,500.33,NaN,NaN,17.87
2012,10962,5480,2677493,"307,415.28",14.13,11.48,488.59,18.50,23.49,21.18
2013,13799,6890,3405860,"408,512.76",14.03,11.99,494.32,27.20,32.89,26.94
2014,17531,8868,4300041,"504,165.97",14.33,11.72,484.89,26.25,23.41,34.01


,line_items,sales,profit,sales_share_pct,margin_pct
order_month,,,,,
Jan,2599,675141,"73,535.38",5.34,10.89
Feb,2168,543768,"72,509.73",4.30,13.33
Mar,3035,770519,"94,087.52",6.09,12.21
Apr,3057,698603,"73,513.63",5.53,10.52
May,3747,904061,"104,509.78",7.15,11.56
Jun,5331,1269751,"147,079.42",10.04,11.58
Jul,3166,749423,"78,070.09",5.93,10.42
Aug,5078,1293852,"151,854.11",10.23,11.74
Sep,5975,1437432,"170,438.18",11.37,11.86


,line_items,sales,profit,sales_share_pct,margin_pct
order_weekday,,,,,
Monday,9177,2235984,"251,746.56",17.69,11.26
Tuesday,9307,2268450,"252,741.92",17.94,11.14
Wednesday,9017,2169290,"242,405.56",17.16,11.17
Thursday,8871,2245934,"278,366.43",17.76,12.39
Friday,9348,2322959,"276,753.97",18.37,11.91
Saturday,4580,1177975,"137,489.07",9.32,11.67
Sunday,990,222313,"29,531.31",1.76,13.28


,line_items,sales,profit,sales_share_pct,margin_pct
order_quarter,,,,,
1,7802,1989428,"240,132.63",15.74,12.07
2,12135,2872415,"325,102.83",22.72,11.32
3,14219,3480707,"400,362.38",27.53,11.50
4,17134,4300355,"503,436.97",34.01,11.71


## 9. Orders and products

`order_id` alone understates the number of real order instances. A composite key of `order_id + customer_name + order_date` produces 25,754 order instances and resolves every cross-country and cross-ship-mode collision. Average order value is $490.91, median order value is lower, and most orders contain one to three lines.

In [11]:
order_view = df.groupby("order_key").agg(
    line_items=("sales", "size"),
    sales=("sales", "sum"),
    profit=("profit", "sum"),
    categories=("category", "nunique")
)
order_view["margin_pct"] = order_view["profit"] / order_view["sales"] * 100

order_summary = pd.Series({
    "composite orders": len(order_view),
    "average order value": order_view["sales"].mean(),
    "median order value": order_view["sales"].median(),
    "p90 order value": order_view["sales"].quantile(0.9),
    "average lines per order": order_view["line_items"].mean(),
    "orders with multiple categories": int((order_view["categories"] > 1).sum()),
    "multi-category order share_pct": (order_view["categories"] > 1).mean() * 100,
})
display(order_summary.to_frame("value").round(2))
display(order_view["sales"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("order_sales").round(2))

product_view = df.groupby(["product_id", "product_name"]).agg(
    sales=("sales", "sum"),
    profit=("profit", "sum"),
    units=("quantity", "sum")
)
product_view["margin_pct"] = product_view["profit"] / product_view["sales"] * 100
display(product_view.sort_values("sales", ascending=False).head(10).round(2))
display(product_view.sort_values("profit", ascending=False).head(10).round(2))

,value
composite orders,"25,754.00"
average order value,490.91
median order value,201.00
p90 order value,"1,283.00"
average lines per order,1.99
orders with multiple categories,"8,742.00"
multi-category order share_pct,33.94


,order_sales
count,"25,754.00"
mean,490.91
std,788.79
min,1.00
25%,62.00
50%,201.00
75%,597.00
90%,"1,283.00"
95%,"1,915.35"
99%,"3,723.00"


,,sales,profit,units,margin_pct
product_id,product_name,,,,
TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61600,"25,199.93",20,40.91
TEC-PH-10004664,"Nokia Smart Phone, with Caller ID",30042,"5,455.95",52,18.16
OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,27454,"7,753.04",31,28.24
TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferencing Unit,22638,"-1,811.08",6,-8.00
TEC-PH-10004823,"Nokia Smart Phone, Full Size",22261,"8,121.48",39,36.48
FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,21870,0.00,39,0.00
FUR-CH-10000027,"SAFCO Executive Leather Armchair, Black",21329,"1,363.23",50,6.39
OFF-AP-10004512,"Hoover Stove, Red",21148,"10,345.58",38,48.92
OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,19824,"2,233.51",37,11.27


,,sales,profit,units,margin_pct
product_id,product_name,,,,
TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61600,"25,199.93",20,40.91
OFF-AP-10004512,"Hoover Stove, Red",21148,"10,345.58",38,48.92
TEC-PH-10004823,"Nokia Smart Phone, Full Size",22261,"8,121.48",39,36.48
OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,27454,"7,753.04",31,28.24
TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,18840,"6,983.88",38,37.07
FUR-CH-10002250,"Office Star Executive Leather Armchair, Black",15289,"6,123.26",37,40.05
TEC-PH-10004664,"Nokia Smart Phone, with Caller ID",30042,"5,455.95",52,18.16
OFF-AP-10002330,"Hamilton Beach Stove, Silver",18248,"5,452.46",38,29.88
TEC-PH-10000303,"Samsung Smart Phone, VoIP",16797,"5,356.81",31,31.89


## 10. Data-quality checks and agent guardrails

The most important risks are not missing values. They are semantic: a non-unique order ID, a customer label that is not a customer ID, mixed date formats, a product name that is not a key, and line-level loss values that are more extreme than negative revenue. These must be handled in code and stated in answers.

In [12]:
raw_sales = pd.read_csv(DATA_PATH, usecols=["sales"], dtype=str)["sales"]
quality_checks = pd.DataFrame([
    {"check": "Missing values in 21 source columns", "value": f"{int(df[original_columns].isna().sum().sum()):,}", "risk": "Low"},
    {"check": "Missing values in engineered columns", "value": f"{int(df[engineered_columns].isna().sum().sum()):,}", "risk": "Low, from zero-sales line"},
    {"check": "Exact duplicate rows", "value": f"{int(df.duplicated().sum()):,}", "risk": "Low"},
    {"check": "Raw sales values with comma separators", "value": f"{int(raw_sales.str.contains(',', regex=False).sum()):,}", "risk": "Medium"},
    {"check": "Distinct order_id vs line items", "value": f"{df['order_id'].nunique():,} vs {len(df):,}", "risk": "High for order metrics"},
    {"check": "Order IDs spanning multiple customers", "value": f"{int((order_rollup['customers'] > 1).sum()):,}", "risk": "High for order metrics"},
    {"check": "Distinct customer_name values", "value": f"{df['customer_name'].nunique():,}", "risk": "High for customer metrics"},
    {"check": "Maximum countries for one customer name", "value": f"{int(customer_country_count.max()):,}", "risk": "High for customer metrics"},
    {"check": "Mixed date separators", "value": f"{int(raw_order_dates.str.contains('/', regex=False).sum()):,} slash / {int(raw_order_dates.str.contains('-', regex=False).sum()):,} dash", "risk": "Medium, resolved day-first"},
    {"check": "Negative ship gaps after day-first parse", "value": f"{int((df['ship_days'] < 0).sum()):,}", "risk": "Low after parsing"},
    {"check": "Year column mismatches order date", "value": f"{int((df['year'] != df['year_check']).sum()):,}", "risk": "Low"},
    {"check": "Zero-sales lines", "value": f"{int((df['sales'] == 0).sum()):,}", "risk": "Low"},
    {"check": "Lines with profit below negative sales", "value": f"{int((df['profit'] < -df['sales']).sum()):,}", "risk": "High for averages"},
    {"check": "Lines with shipping_cost above sales", "value": f"{int((df['shipping_cost'] > df['sales']).sum()):,}", "risk": "Low"},
    {"check": "Duplicate order_id + product_id rows", "value": f"{int(df.duplicated(['order_id', 'product_id']).sum()):,}", "risk": "Low to Medium"},
    {"check": "State names appearing in multiple countries", "value": f"{int((state_country_count > 1).sum()):,}", "risk": "Medium"},
    {"check": "Product names mapping to multiple product IDs", "value": f"{int((product_name_id_count > 1).sum()):,}", "risk": "Medium"},
    {"check": "Currency column present", "value": "No", "risk": "Medium for cross-country money"},
])
display(quality_checks)

metric_contract = pd.DataFrame([
    {"metric": "Line-item sales", "definition": "SUM(sales) on the raw row grain", "guardrail": "Do not deduplicate by order_id"},
    {"metric": "Order sales", "definition": "SUM(sales) grouped by order_key", "guardrail": "order_id alone collides across customers"},
    {"metric": "Profit margin", "definition": "SUM(profit) / SUM(sales)", "guardrail": "Never average line-level margins"},
    {"metric": "Loss rate", "definition": "COUNT(profit < 0) / COUNT(*)", "guardrail": "State that this is line-item based"},
    {"metric": "Average discount", "definition": "AVG(discount) at line grain", "guardrail": "Do not call it an order-level discount"},
    {"metric": "Shipping ratio", "definition": "SUM(shipping_cost) / SUM(sales)", "guardrail": "Use aggregate sums, not row ratios"},
    {"metric": "Customer count", "definition": "Not supported", "guardrail": "customer_name is a reused label, not a customer ID"},
    {"metric": "Product count", "definition": "COUNT(DISTINCT product_id)", "guardrail": "product_name is not unique"},
    {"metric": "Geography key", "definition": "country + state", "guardrail": "state alone is not globally unique"},
    {"metric": "Money comparisons", "definition": "Treat all money as one common currency", "guardrail": "No currency column exists"},
])
display(metric_contract)

,check,value,risk
0,Missing values in 21 source columns,0,Low
1,Missing values in engineered columns,2,"Low, from zero-sales line"
2,Exact duplicate rows,0,Low
3,Raw sales values with comma separators,"2,630",Medium
4,Distinct order_id vs line items,"25,035 vs 51,290",High for order metrics
5,Order IDs spanning multiple customers,659,High for order metrics
6,Distinct customer_name values,795,High for customer metrics
7,Maximum countries for one customer name,32,High for customer metrics
8,Mixed date separators,"20,067 slash / 31,223 dash","Medium, resolved day-first"
9,Negative ship gaps after day-first parse,0,Low after parsing


,metric,definition,guardrail
0,Line-item sales,SUM(sales) on the raw row grain,Do not deduplicate by order_id
1,Order sales,SUM(sales) grouped by order_key,order_id alone collides across customers
2,Profit margin,SUM(profit) / SUM(sales),Never average line-level margins
3,Loss rate,COUNT(profit < 0) / COUNT(*),State that this is line-item based
4,Average discount,AVG(discount) at line grain,Do not call it an order-level discount
5,Shipping ratio,SUM(shipping_cost) / SUM(sales),"Use aggregate sums, not row ratios"
6,Customer count,Not supported,"customer_name is a reused label, not a customer ID"
7,Product count,COUNT(DISTINCT product_id),product_name is not unique
8,Geography key,country + state,state alone is not globally unique
9,Money comparisons,Treat all money as one common currency,No currency column exists


## 11. Key takeaways for the agent

- Use the **line-item grain** for sales and profit. The file has 51,290 lines, 25,035 raw order IDs, and 25,754 composite order keys.
- Do **not** use `order_id` alone for order counts or average order value: 659 order IDs span multiple customers and 58 span multiple countries.
- Do **not** count customers from `customer_name`: only 795 names cover 147 countries, and one name appears in 32 countries.
- Parse dates **day-first**; the mix of `1/1/2011` and `13-01-2011` makes month-first parsing produce thousands of impossible shipping gaps.
- Parse `sales` with thousands separators. Two thousand six hundred thirty raw values contain commas.
- Discount is the dominant profit lever: lines with zero discount are never loss-making, while lines with 50%+ discount are 99.7% loss-making.
- Furniture is the weak category because Tables has a -8.47% margin and 57.61% of its lines lose money.
- APAC is the largest market, but EMEA has the weakest large-market margin; several countries are structurally negative.
- Growth is volume-led: sales rise every year, while margin stays around 11%-12%.
- Flag the 2,260 lines where profit is below negative sales as anomalies or return-like effects before using averages.
- There is no currency column, so cross-country monetary comparisons require an explicit assumption.

In [13]:
findings = [
    f"{len(df):,} line items across {df['order_key'].nunique():,} composite orders, {df['country'].nunique()} countries, {df['product_id'].nunique():,} product IDs and {df['customer_name'].nunique()} customer labels.",
    f"Sales are ${df['sales'].sum():,.0f}, profit is ${df['profit'].sum():,.0f}, and aggregate margin is {df['profit'].sum() / df['sales'].sum() * 100:.2f}%.",
    f"Loss-making lines are {((df['profit'] < 0).mean() * 100):.2f}% of rows and total ${df.loc[df['profit'] < 0, 'profit'].sum():,.0f}.",
    f"Lines with 50%+ discount are {((df['discount'] >= 0.5).mean() * 100):.2f}% of rows and {((df.loc[df['discount'] >= 0.5, 'profit'] < 0).mean() * 100):.2f}% of them lose money.",
    f"Tables is the only loss-making sub-category, with a {subcategory_view.loc['Tables', 'margin_pct']:.2f}% margin and a {subcategory_view.loc['Tables', 'loss_share_pct']:.2f}% loss rate.",
    f"EMEA margin is {market_view.loc['EMEA', 'margin_pct']:.2f}%, versus APAC at {market_view.loc['APAC', 'margin_pct']:.2f}%.",
    f"2014 contributes {yearly.loc[2014, 'sales_share_pct']:.2f}% of sales, and annual sales growth is {yearly.loc[2014, 'sales_yoy_pct']:.2f}%.",
    f"{int((df['profit'] < -df['sales']).sum()):,} lines have profit below negative sales, so averages must be reported with a sensitivity check.",
]
for finding in findings:
    print(f"- {finding}")

- 51,290 line items across 25,754 composite orders, 147 countries, 10,292 product IDs and 795 customer labels.
- Sales are $12,642,905, profit is $1,469,035, and aggregate margin is 11.62%.
- Loss-making lines are 24.46% of rows and total $-920,357.
- Lines with 50%+ discount are 11.32% of rows and 99.71% of them lose money.
- Tables is the only loss-making sub-category, with a -8.47% margin and a 57.61% loss rate.
- EMEA margin is 5.45%, versus APAC at 12.20%.
- 2014 contributes 34.01% of sales, and annual sales growth is 26.25%.
- 2,260 lines have profit below negative sales, so averages must be reported with a sensitivity check.


<StringArray>
['Cisco TelePresence System EX90 Videoconferencing Unit', 'Canon imageCLASS 2200 Advanced Copier']
Length: 2, dtype: str

## 12. Recommended next steps

1. Build the agent's semantic layer around the metric contract above, not around raw column names.
2. Add validation tools that reject month-first date parsing, unweighted margin averages, and customer counts based on `customer_name`.
3. Segment discount analysis at 0%, 10%-20%, 30%+, and 50%+ thresholds; the profit breakpoint is already visible in the data.
4. Treat Tables, EMEA, Nigeria, Turkey and the Netherlands as diagnostic slices for loss-driver analysis.
5. Separate the 2,260 profit-below-negative-sales lines into an anomaly bucket before publishing averages or rankings.
6. Add a currency assumption to every cross-country monetary answer until a currency field is available.